In [1]:
from math import inf
from typing import Any, Dict, List, Tuple


def solve_carbon_dp(
    configs: List[Any],
    carbon_intensity: List[float],
    power: Dict[Any, float],
    speed: Dict[Any, int],
    work_required: int,
    switch_cost: float,
):
    """
    Solve the carbon-aware parallel configuration scheduling problem exactly
    using dynamic programming.

    Objective:
        minimize sum_t P(pi_t) * CI(t)
              + switch_cost * number_of_switches

    Subject to:
        total work >= work_required

    Args:
        configs:
            List of valid parallel configs, e.g.
            [(4, 1, 1), (1, 4, 1), (1, 1, 4), ...]

        carbon_intensity:
            CI[t] values for t = 0, ..., T-1.

        power:
            power[config] = steady-state power draw of config.

        speed:
            speed[config] = work completed per time step.
            This should be an integer number of steps/tokens/chunks.

        work_required:
            Total amount of work W that must be completed.

        switch_cost:
            Constant cost paid whenever pi_t != pi_{t-1}.

    Returns:
        best_cost:
            Minimum carbon cost.

        best_schedule:
            List of configurations, one per time step.

        best_work:
            Total work completed by the returned schedule.
    """

    T = len(carbon_intensity)
    max_speed = max(speed[c] for c in configs)
    max_work = work_required + max_speed

    # dp[t][w][c] = best cost after time t, with w work, ending in config c
    dp = {}
    parent = {}

    # Initialization at t = 0
    for c in configs:
        w = min(speed[c], max_work)
        cost = power[c] * carbon_intensity[0]
        dp[(0, w, c)] = cost
        parent[(0, w, c)] = None

    # Dynamic programming recurrence
    for t in range(1, T):
        new_dp = {}

        for (prev_t, prev_w, prev_c), prev_cost in dp.items():
            if prev_t != t - 1:
                continue

            for c in configs:
                new_w = min(prev_w + speed[c], max_work)

                run_cost = power[c] * carbon_intensity[t]
                transition_cost = switch_cost if c != prev_c else 0.0

                total_cost = prev_cost + run_cost + transition_cost

                state = (t, new_w, c)

                if total_cost < new_dp.get(state, inf):
                    new_dp[state] = total_cost
                    parent[state] = (t - 1, prev_w, prev_c)

        dp.update(new_dp)

    # Find best final state with enough work
    best_state = None
    best_cost = inf

    for (t, w, c), cost in dp.items():
        if t == T - 1 and w >= work_required:
            if cost < best_cost:
                best_cost = cost
                best_state = (t, w, c)

    if best_state is None:
        raise ValueError("No feasible schedule completes the required work.")

    # Recover optimal schedule
    schedule = []
    state = best_state

    while state is not None:
        t, w, c = state
        schedule.append(c)
        state = parent[state]

    schedule.reverse()

    best_work = best_state[1]

    return best_cost, schedule, best_work

In [4]:
slot_seconds = 60.0
average_reshard_wall_time_s = 22.33

strategies = [
    {"tp": 1, "pp": 1, "dp": 4, "power_w": 729.9, "step_time_s": 0.2094},
    {"tp": 4, "pp": 1, "dp": 1, "power_w": 664.1, "step_time_s": 0.8995},
    {"tp": 1, "pp": 4, "dp": 1, "power_w": 644.2, "step_time_s": 0.2908},
    {"tp": 2, "pp": 2, "dp": 1, "power_w": 561.2, "step_time_s": 0.5162},
    {"tp": 2, "pp": 1, "dp": 2, "power_w": 584.2, "step_time_s": 0.4492},
    {"tp": 1, "pp": 2, "dp": 2, "power_w": 664.6, "step_time_s": 0.2635},
]

configs = [
    (s["dp"], s["tp"], s["pp"])
    for s in strategies
]

# Power cost per 60-second time slot, in kWh.
power_kw = {
    (s["dp"], s["tp"], s["pp"]): (s["power_w"] / 1000.0) * (slot_seconds / 3600.0)
    for s in strategies
}

# Work per 60-second time slot, measured in optimization steps.
speed = {
    (s["dp"], s["tp"], s["pp"]): slot_seconds / s["step_time_s"]
    for s in strategies
}

# Example carbon intensity trace, in gCO2/kWh.
carbon_intensity = [
    139, 133, 129, 123, 122, 119,
    119, 114, 77, 60, 56, 55,
    54, 54, 55, 60, 72, 90,
]

# Constant switching cost.
# Units: gCO2.
# Approximate it as:
# average power during reshard * reshard duration * average carbon intensity.
avg_power_kw = sum(s["power_w"] for s in strategies) / len(strategies) / 1000.0
avg_ci = sum(carbon_intensity) / len(carbon_intensity)

switch_cost = avg_power_kw * (average_reshard_wall_time_s / 3600.0) * avg_ci

work_required = 2500  # required optimization steps

best_cost, schedule, work_done = solve_carbon_dp(
    configs=configs,
    carbon_intensity=carbon_intensity,
    power_kw=power_kw,
    speed=speed,
    work_required=work_required,
    switch_cost=switch_cost,
)

print("Best carbon cost:", best_cost, "gCO2")
print("Work done:", work_done, "steps")
print("Switch cost:", switch_cost, "gCO2")
print("Schedule:")

for t, config in enumerate(schedule):
    print(f"t={t:02d}, CI={carbon_intensity[t]}, config={config}, steps={speed[config]:.1f}")

Best carbon cost: 16.07364409744341 gCO2
Work done: 2538 steps
Switch cost: 0.3604740974434156 gCO2
Schedule:
t=00, CI=139, config=(1, 2, 2), steps=116.2
t=01, CI=133, config=(1, 2, 2), steps=116.2
t=02, CI=129, config=(1, 2, 2), steps=116.2
t=03, CI=123, config=(1, 2, 2), steps=116.2
t=04, CI=122, config=(1, 2, 2), steps=116.2
t=05, CI=119, config=(1, 2, 2), steps=116.2
t=06, CI=119, config=(1, 2, 2), steps=116.2
t=07, CI=114, config=(1, 2, 2), steps=116.2
t=08, CI=77, config=(1, 2, 2), steps=116.2
t=09, CI=60, config=(1, 2, 2), steps=116.2
t=10, CI=56, config=(1, 2, 2), steps=116.2
t=11, CI=55, config=(1, 2, 2), steps=116.2
t=12, CI=54, config=(1, 2, 2), steps=116.2
t=13, CI=54, config=(1, 1, 4), steps=206.3
t=14, CI=55, config=(1, 1, 4), steps=206.3
t=15, CI=60, config=(1, 1, 4), steps=206.3
t=16, CI=72, config=(1, 1, 4), steps=206.3
t=17, CI=90, config=(1, 1, 4), steps=206.3
